In [ ]:
import numpy as np
from itertools import combinations
import os

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
from matplotlib import colors
%matplotlib inline

In [ ]:
import pandas as pd
import numpy as np
from multiprocessing import Pool

In [ ]:
import bioframe
import cooler
import cooltools

In [ ]:
resolution = 100
A = cooler.Cooler('PATH_TO/A.mcool::/resolutions/'+str(resolution))
B = cooler.Cooler('PATH_TO/B.mcool::/resolutions/'+str(resolution))

In [ ]:
#loading the chromosome size file
chromsizes = pd.read_csv('PATH_TO/chrom_size.csv')
chromsizes.head()

In [ ]:
chromsizes
bioframe.core.checks.is_viewframe(chromsizes)

In [ ]:
#this is calculating the smoothed curves for everything
cvd = cooltools.expected_cis(
    clr=A,
    view_df=chromsizes,
    smooth=True,
    aggregate_smoothed=True,
    smooth_sigma=0.1,
    nproc=1
)

cvd2 = cooltools.expected_cis(
    clr=B,
    view_df=chromsizes,
    smooth=True,
    aggregate_smoothed=True,
    smooth_sigma=0.1,
    nproc=1
)


In [ ]:
#plotting it
cvd['balanced.avg.smoothed'].loc[cvd['dist'] < 2] = np.nan
cvd2['balanced.avg.smoothed'].loc[cvd2['dist'] < 2] = np.nan

f, ax = plt.subplots(1,1)

for region in chromsizes['name']:
    ax.loglog(
        cvd['dist_bp'].loc[cvd['region1']==region],
        cvd['balanced.avg.smoothed'].loc[cvd['region1']==region],
        cvd2['dist_bp'].loc[cvd2['region1']==region],
        cvd2['balanced.avg.smoothed'].loc[cvd2['region1']==region],
    )
    ax.set(
        xlabel='separation, bp',
        ylabel='IC contact frequency')
    ax.set_aspect(1.0)
    ax.grid(lw=0.5)


In [ ]:
#calculate the derivatives
cvd_merged = cvd.drop_duplicates(subset=['dist'])[['dist_bp', 'balanced.avg.smoothed.agg']]
cvd2_merged = cvd2.drop_duplicates(subset=['dist'])[['dist_bp', 'balanced.avg.smoothed.agg']]
cvd4_merged = cvd4.drop_duplicates(subset=['dist'])[['dist_bp', 'balanced.avg.smoothed.agg']]

In [ ]:
#calculate the derivatives
der = np.gradient(np.log(cvd_merged['balanced.avg.smoothed.agg']),
                  np.log(cvd_merged['dist_bp']))
der2 = np.gradient(np.log(cvd2_merged['balanced.avg.smoothed.agg']),
                  np.log(cvd2_merged['dist_bp']))
der4 = np.gradient(np.log(cvd4_merged['balanced.avg.smoothed.agg']),
                  np.log(cvd4_merged['dist_bp']))

In [ ]:
f, axs = plt.subplots(
    figsize=(6.5,13),
    nrows=2,
    gridspec_kw={'height_ratios':[6,2]},
    sharex=True)
ax = axs[0]
ax.loglog(
    cvd_merged['dist_bp'],
    cvd_merged['balanced.avg.smoothed.agg'],
    cvd2_merged['dist_bp'],
    cvd2_merged['balanced.avg.smoothed.agg'],
    cvd4_merged['dist_bp'],
    cvd4_merged['balanced.avg.smoothed.agg'],
    '-'
)

ax.set(
    ylabel='IC contact frequency',
    xlim=(4e2,1e7),
    ylim=(1e-6,1e-1)
)
ax.set_aspect(1.0)
ax.grid(lw=0.5)


ax = axs[1]
ax.semilogx(
    cvd_merged['dist_bp'],
    der,
    cvd2_merged['dist_bp'],
    der2,

    alpha=0.5
)

ax.set(
    xlabel='separation, bp',
    ylabel='slope')

ax.grid(lw=0.5)

ax.legend(['title_A', 'title_B'], loc="center left", bbox_to_anchor=(1, 0.5))
plt.rcParams['pdf.fonttype'] = 'truetype'
plt.rcParams['svg.fonttype'] = 'none' # No text as paths. Assume font installed.
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['text.usetex'] = False
plt.savefig('output_file_name.svg', bbox_inches='tight')